# Part II · Lesson 7 — Watch a recipe appear

### Import the graph helpers

In [ ]:
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END

### Import the model and messages

In [ ]:
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage, SystemMessage

### Connect to our existing Groq model

In [ ]:
# Keep GROQ_API_KEY in the local .env file.
load_dotenv()
model = init_chat_model(
    "openai/gpt-oss-120b",
    model_provider="groq",
    reasoning_effort="low",
)

### Describe the recipe record

In [ ]:
class RecipeState(TypedDict):
    ingredients: list[str]
    servings: int
    request: str
    recipe: str
    status: str

### Prepare a clear recipe request

In [ ]:
def prepare_request(state: RecipeState) -> dict:
    ingredients = ", ".join(state["ingredients"])
    request = (
        f"Suggest a quick vegetarian dinner for {state['servings']} people "
        f"using {ingredients}. Give a short recipe with numbered steps."
    )
    return {"request": request, "status": "ingredients prepared"}

### Generate the recipe

In [ ]:
def write_recipe(state: RecipeState) -> dict:
    reply = model.invoke([
        SystemMessage("You are a concise recipe assistant."),
        HumanMessage(state["request"]),
    ])
    return {"recipe": reply.text, "status": "recipe ready"}

### Connect the two-node graph

In [ ]:
builder = StateGraph(RecipeState)
builder.add_node("prepare_request", prepare_request)
builder.add_node("write_recipe", write_recipe)
builder.add_edge(START, "prepare_request")
builder.add_edge("prepare_request", "write_recipe")
builder.add_edge("write_recipe", END)
graph = builder.compile()
recipe_input = {
    "ingredients": ["rice", "peas", "carrots"], "servings": 2,
    "request": "", "recipe": "", "status": "new",
}

### Stream completed node updates

In [ ]:
for event in graph.stream(recipe_input, stream_mode="updates"):
    for node_name, update in event.items():
        print("Finished node:", node_name)
        print(update)

### Stream the model's arriving text

In [ ]:
for chunk, metadata in graph.stream(recipe_input, stream_mode="messages"):
    if metadata["langgraph_node"] == "write_recipe":
        print(chunk.text, end="", flush=True)
print()